## 05 - FrontEnd Architecture

Purpose: build realistic frontend and test / optimize

In [1]:
# Imports

import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt
import soundfile as sf
import time
from scipy.signal import firwin

from util import sigmoid_squash, pchirp, get_bw_actual, bw_fc_to_f1f2
from util import MAX_BW, MIN_BW, MAX_DEGREE, MIN_DEGREE, MAX_DUR_MS, MIN_DUR_MS, MIN_RATIO, MIN_FREQ, MAX_FREQ, FS

%matplotlib widget

In [2]:
# Define ChirpletFilterbank class
#    Note that learnable parameters theta_<param> get mapped to the actual inputs that create chirplets after passing though softplus/sigmoid for continuity

class ChirpletFilterbank(nn.Module):

    def __init__(self, n_channels):
        super().__init__()

        # Parameters
        self.n_channels = n_channels

        # Parameter constraints
        self.MAX_BW = MAX_BW        
        self.MIN_BW = MIN_BW
        self.MAX_DEGREE = MAX_DEGREE
        self.MIN_DEGREE = MIN_DEGREE
        self.MAX_DUR_MS = MAX_DUR_MS
        self.MIN_DUR_MS = MIN_DUR_MS
        self.MIN_RATIO = MIN_RATIO
        self.MAX_FREQ = MAX_FREQ
        self.MIN_FREQ = MIN_FREQ
        self.FS = FS

        # Learnable parameter registration
        self.theta_bw = nn.Parameter(torch.zeros(n_channels))
        self.theta_fc = nn.Parameter(torch.zeros(n_channels))
        self.theta_T = nn.Parameter(torch.zeros(n_channels))
        self.theta_degree = nn.Parameter(torch.zeros(n_channels))
        self.theta_sign = nn.Parameter(torch.zeros(n_channels))
        

    def _get_constrained_params(self):
        """
        Apply warping so that continuous theta_* parameters respect constraints before being fed as chirplet gen inputs

        Take self.theta_* and return constrained version

        Internal method, called only by forward()
        """

        bw = sigmoid_squash(self.theta_bw, self.MIN_BW, self.MAX_BW)

        # fc's valid range depends on bw
        fc_lo = self.MIN_FREQ + bw / 2
        fc_hi = torch.min(self.MAX_FREQ - bw / 2, bw * (self.MIN_RATIO + 1) / (2 * (self.MIN_RATIO - 1)))
        fc = sigmoid_squash(self.theta_fc, fc_lo, fc_hi)

        T = sigmoid_squash(self.theta_T, self.MIN_DUR_MS, self.MAX_DUR_MS)
        degree = sigmoid_squash(self.theta_degree, self.MIN_DEGREE, self.MAX_DEGREE)

        return (bw, fc, T, degree)
                               
    def _generate_kernels(self, bw, fc, T, degree):
        """
        Generate chirplet kernels from parameter tensors
        """

        # Compute f1, f2 from bw and fc tensors
        f1 = fc - bw / 2
        f2 = fc + bw / 2

        # Incorporate sweep direction
        w = torch.sigmoid(self.theta_sign)
        snap = (w > 0.5).float().detach() # snaps to 0 or 1 but removes grad flow
        disc = snap + (w - w.detach()) # allows for w gradient flow in backprop
        f_start = disc * f1 + (1 - disc) * f2
        f_end = disc * f2 + (1 - disc) * f1

        # Time vector (u = t / T)
        Tmax = self.MAX_DUR_MS / 1000.
        t = torch.arange(int(Tmax * self.FS), device=bw.device)/ self.FS # keep this tensor on same device as others
        t_b = t.unsqueeze(0) # for broadcasting
        T_b = T.unsqueeze(1) / 1000 # for broadcasting
        mask = (t_b <= T_b).float() # get valid range based on variable T per chan
        u =  t_b / T_b # normalized time arg 
        u = torch.clamp(u, max=1.0)   # guard against overshoot

        # Inst freq
        fsb = f_start.unsqueeze(1)
        feb = f_end.unsqueeze(1)
        degb = degree.unsqueeze(1)
        f_i = (fsb + (feb - fsb) * u ** degb) * mask

        # Inst phase - cumulative trapezoidal integration
        dt = 1.0 / self.FS
        phi_i = 2 * np.pi * dt * torch.cumsum((f_i[:, :-1] + f_i[:, 1:]) / 2, dim=1)
        phi_i = F.pad(phi_i, (1, 0))  # prepend 0, matching cumulative_trapezoid's initial=0
        
        # Variable length hann window per channel
        win = 0.5 * (1. - torch.cos(2 * np.pi * u)) * mask

        return torch.sin(phi_i) * win

    def forward(self, x):
        """
        Forward propagate

        x: [batch, 1, signal_len]
        """

        # Get kernel gen params
        bw, fc, T, degree = self._get_constrained_params()

        # Create kernels, reshape, and time reverse for actual conv
        k = self._generate_kernels(bw, fc, T, degree) # out: [n_channels, kernel_len]
        k = k.unsqueeze(1) # conv1d expects [out_ch, in_ch, kernel_len]
        k = k.flip(-1)

        # Conv
        y = F.conv1d(x, k, padding='same')

        # return both y and k
        return (y, k.flip(-1).squeeze(1))

In [4]:
# Define BLR class (Band limited resampling)
#   No learnable params, just transforms and compresses inputs

class BLR(nn.Module):

    def __init__(self):
        super().__init__()

        self.FS = FS
        self.BW = 2400
        self.up = 3   # 16k -> 2.4k
        self.down = 20 # 16k -> 2.4k

        # Define FIR LPF for resample poly
        cutoff = 1.0 / self.down
        n_taps = 20 * self.down + 1
        filt = firwin(n_taps, cutoff, window=('kaiser', 5.0)) * self.up   # amplitude scaling required due to zero stuffing
        self.register_buffer('fir_filter', torch.tensor(filt, dtype=torch.float32))
        

    def _hilbert_torch(self, x):
        """
        torch tensor compatible hilbert transform

        x: [...., signal_len]
        """
        N = x.shape[-1]
        X = torch.fft.fft(x, dim=-1) # transform along final dim
        
        h = torch.zeros(N, device=x.device, dtype=x.dtype) # hilbert spectrum mask
        h[0] = 1.
        if N % 2 == 0:
            h[N//2] = 1
            h[1:N//2] = 2
        else:
            h[1:(N+1)//2] = 2

        return torch.fft.ifft(X * h, dim=-1)

    def _resample_poly_torch(self, x):
        """
        torch tensor compatible resample_poly

        x: [..... signal_len], complex
        """
        # Get dims
        *batch_dim, L = x.shape

        # Split real / imag
        xr = x.real
        xi = x.imag

        # Zero stuff by "up" factor
        xr_up = torch.zeros(*batch_dim, L * self.up, dtype=xr.dtype, device=x.device)
        xr_up[..., ::self.up] = xr
        xi_up = torch.zeros(*batch_dim, L * self.up, dtype=xi.dtype, device=x.device)
        xi_up[..., ::self.up] = xi

        # Conv1d to apply fir filter + necessary reshaping
        N = int(np.prod(batch_dim))
        xr_flat = xr_up.reshape(N, 1, L * self.up)
        xi_flat = xi_up.reshape(N, 1, L * self.up)
        x_stack = torch.cat([xr_flat, xi_flat], dim=0)   # [2*B*C, 1, L*up]
        
        filt = self.fir_filter.view(1, 1, -1)
        x_filt = F.conv1d(x_stack, filt, padding=self.fir_filter.shape[0] // 2)
        
        xr_filt, xi_filt = x_filt.chunk(2, dim=0)
        xr_filt = xr_filt.reshape(*batch_dim, -1)
        xi_filt = xi_filt.reshape(*batch_dim, -1)

        # Decimate
        xr_dec = xr_filt[..., ::self.down]
        xi_dec = xi_filt[..., ::self.down]

        # Trim
        new_len = int(self.BW * (L / self.FS))
        assert new_len <= xr_dec.shape[-1], "Not enough samples post decimation"
        xr_dec = xr_dec[..., :new_len]
        xi_dec = xi_dec[..., :new_len]

        return torch.complex(xr_dec, xi_dec)

    def compute_fce_batch(self, kernels, threshold_db=-40):
        """
        Compute empirical band-center frequency per channel via -40dB edge detection.
        Intentionally non-differentiable: fce is a fixed empirical correction,
        not a learned quantity — used only to center demodulation.
        """
        with torch.no_grad():
            X = torch.fft.rfft(kernels, dim=-1)
            mag_db = 20*torch.log10(torch.abs(X) + 1e-12)
            mag_db -= mag_db.max(dim=-1, keepdim=True).values
            above = mag_db > threshold_db
            freqs = torch.fft.rfftfreq(kernels.shape[-1], 1/self.FS).to(kernels.device)
            idx = torch.arange(above.shape[-1], device=kernels.device)
            first_idx = torch.where(above, idx, idx.max()).min(dim=-1).values
            last_idx  = torch.where(above, idx, idx.min()).max(dim=-1).values
            f1e = freqs[first_idx]
            f2e = freqs[last_idx]
            return (f1e + f2e) / 2
        
    def forward(self, x, kernels):
        """
        x: [B, C, L], real, output of ChirpletFilterbank
        kernels: [C, kernel_len], chirplet kernels (for fce computation)
        """

        # Handle length internally if given input isn't already properly dimensioned
        L = x.shape[-1]
        trim_len = L - (L % self.down)
        if trim_len < L:
            x = x[..., :trim_len]
            L = trim_len
            
        xh = self._hilbert_torch(x)  # [B, C, L], complex
    
        fce = self.compute_fce_batch(kernels)  # [C]
    
        L = x.shape[-1]
        t = torch.arange(L, device=x.device, dtype=torch.float32) / self.FS  # [L]
        t_b = t.view(1, 1, L)
        fce_b = fce.view(1, -1, 1)  # [1, C, 1]
    
        xd = xh * torch.exp(-1j * 2 * torch.pi * fce_b * t_b)
    
        return self._resample_poly_torch(xd)
        

In [ ]:
# Full AudioFrontEnd class

class AudioFrontEnd(nn.Module):

    def __init__(self, n_channels):
        super().__init__()

        self.chirplet_bank = ChirpletFilterbank(n_channels)
        self.blr = BLR()

    def forward(self, x):
        """
        Run the audio front end: chirplet filter bank -> band limited resampling

        x: [batch, signal_len] or [batch, 1, signal_len]
        """
        # Reshape x to 3D if needed
        if x.dim() == 2:
            x = x.unsqueeze(1)   # -> [batch, 1, signal_len]

        # Trim signal_len to multiple of 20 if needed (inputs assumed ~4s at 16kHz)
        L = x.shape[-1]
        trim_len = L - (L % self.blr.down)
        if trim_len < L:
            x = x[..., :trim_len]

        # Apply filter bank
        y, k = self.chirplet_bank(x)   # y: [B, C, L], real  k: [C, kernel_len]

        return self.blr(y, k)